# Enrich D4GC document metadata

This pipeline reads `data/documentation.csv` and the parsed Markdown corpus. It writes `data/processed/documentation_enriched.csv`, leaving the source untouched. Geography fields are **mentions in text**, not verified primary locations. The curated dictionaries are in `enrich_metadata.py` and are deliberately incomplete. Review matches before using them as RAG filters.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'documentation.csv').exists())
sys.path.insert(0, str(repo / 'src'))
from msf_navigator.enrich_metadata import run, OUTPUT
rows = run()
df = pd.DataFrame(rows)
print(f'Wrote {len(df)} rows to {OUTPUT}')
print('Columns:', ', '.join(df.columns))

## Coverage and quality checks

The row count and original columns must match the source. Unavailable documents remain in the output with blank enrichment fields. Exact duplicate text is flagged for later deduplication; all document IDs are retained.

In [ ]:
source = pd.read_csv(repo / 'data' / 'documentation.csv', dtype=str).fillna('')
assert len(df) == len(source)
assert df[['id','genre','language']].reset_index(drop=True).equals(source[['id','genre','language']].reset_index(drop=True))
assert df['id'].is_unique
assert (df.loc[df.parsed_available.eq('false'), ['country_mentions','city_mentions','topic_tags']] == '').all().all()
print('Rows:', len(df), 'parsed:', df.parsed_available.eq('true').sum(), 'missing parsed:', df.parsed_available.eq('false').sum())
print('Country mentions:', df.country_mentions.ne('').sum(), 'city mentions:', df.city_mentions.ne('').sum())
print('Duplicate text rows:', pd.to_numeric(df.duplicate_text_group_size, errors='coerce').gt(1).sum())

## Review examples

Inspect evidence snippets to identify publication datelines, broad surveys, ambiguous place names, and missed languages. A document may have several countries or cities separated by `|`. Empty fields mean no lexicon match.

In [ ]:
display(df.loc[df.country_mentions.ne(''), ['id','language','country_mentions','city_mentions','country_evidence']].head(15))
display(df.loc[df.city_mentions.ne(''), ['id','city_mentions','city_evidence']].head(10))

## RAG use and limits

Use `id` and paths as provenance. Treat topic tags and place mentions as candidate filters or facets, with an option to search without them. The current lexicons do not cover every location, Arabic names, synonyms, or transliterations. Broad terms and publication datelines can create false positives. Before production, manually label a sample across languages and genres, extend the gazetteer, and measure mention precision and recall. Do not use these columns to make clinical decisions without inspecting source documents.